# GPT 퀘스트 - Decoder-only Transformer 구현

**날짜:** 2026년 3월 6일  
**목표:** 기존 Encoder-Decoder Transformer를 GPT(Decoder-only) 구조로 변환하여 프리트레인 수행

---
## 1번 문제: Transformer와 GPT의 아키텍처 차이점

### 변경 사항 요약

| 항목 | 기존 Transformer (Encoder-Decoder) | GPT (Decoder-only) |
|------|-----------------------------------|-----------------------|
| 구조 | Encoder + Decoder | Decoder만 사용 |
| Attention | Self-Attn + Cross-Attn (Encoder-Decoder Attn) | Masked Self-Attn만 사용 |
| 위치 정보 | Sinusoidal Positional Encoding (고정) | Positional Embedding (학습 가능, `nn.Embedding`) |
| 학습 방식 | Q→A 정답 쌍 필요 | 단일 문장, Next Token Prediction |
| 데이터 형식 | `(질문, 답변)` 쌍 | `<start> Q <sep> A <end>` 단일 시퀀스 |

### 코드 변경 핵심

**① Encoder 제거 — Decoder-only 구조**
```python
# 기존: Encoder + Decoder
self.transformer_encoder = nn.TransformerEncoder(...)
self.transformer_decoder = nn.TransformerDecoder(...)

# 변경: TransformerEncoder를 Decoder-only로 사용 (causal mask로 autoregressive 보장)
self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
```

**② Sinusoidal → Learnable Positional Embedding**
```python
# 기존: Sine/Cosine 함수 기반 고정 인코딩
self.positional_encoding = self._get_positional_encoding(d_model, max_len)

# 변경: 학습 가능한 임베딩
self.position_embedding = nn.Embedding(max_seq_length, d_model)
```

**③ Dataset 변경 — Next Token Prediction**
```python
# 기존: Q/A 쌍
return {'question': q_tensor, 'answer': a_tensor}

# 변경: 단일 시퀀스, input/target shift
sequence = [<start>] + Q_tokens + [<sep>] + A_tokens + [<end>]
input_ids  = sequence[:-1]
target_ids = sequence[1:]   # next token prediction
```

---
## 2번 문제: 데이터 전처리

- Decoder 기반 생성모델에 맞게 챗봇 데이터를 변환
- `<start> Q <sep> A <end>` 형식으로 delimiter를 붙여 단일 시퀀스 구성
- 이번 과제는 pretrain을 위한 데이터이므로 정답 쌍 불필요

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import pickle

# 데이터 로드
df = pd.read_csv('./data/processed/augmented_data.csv')
print(f'데이터 크기: {len(df)}')
print(df.head())

In [ ]:
# 코퍼스 및 토크나이저 로드
with open('./data/processed/corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

with open('./models/tokenizer.pkl', 'rb') as f:
    tok = pickle.load(f)

word2idx = tok['word2idx'] if isinstance(tok, dict) else tok.word2idx
print(f'어휘 크기: {len(word2idx)}')
print(f'특수 토큰 확인: <start>={word2idx.get("<start>")}, <sep>={word2idx.get("<sep>")}, <end>={word2idx.get("<end>")}')

In [ ]:
# GPTDataset 시퀀스 구성 예시
sample_q = '오늘 기분이 어때'
sample_a = '좋아요 감사합니다'

sequence = ['<start>'] + sample_q.split() + ['<sep>'] + sample_a.split() + ['<end>']
print('시퀀스:', sequence)
print('input_ids:', sequence[:-1])
print('target_ids:', sequence[1:])

---
## 3번 문제: 위치 정보 추가 (Positional Embedding)

- 기존 Sinusoidal Encoding → 학습 가능한 `nn.Embedding`으로 변경
- 데이터의 위치 정보를 토큰 임베딩에 더하여 순서 정보 제공

In [ ]:
import torch
import torch.nn as nn
import importlib.util

spec = importlib.util.spec_from_file_location('train', './scripts/04_train.py')
tm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tm)

import config

vocab_size = len(word2idx)
model = tm.GPTModel(
    vocab_size=vocab_size,
    d_model=config.TRANSFORMER_D_MODEL,
    nhead=config.TRANSFORMER_NHEAD,
    num_layers=config.TRANSFORMER_NUM_LAYERS,
    dim_feedforward=config.TRANSFORMER_DIM_FEEDFORWARD,
    dropout=0.0,
    max_seq_length=config.MAX_SEQ_LENGTH
)

# Positional Embedding 확인
print('Position Embedding 타입:', type(model.position_embedding))
print('Position Embedding 크기:', model.position_embedding.weight.shape)
print('→ (max_seq_length=50, d_model=768) — 학습 가능한 파라미터')

---
## 4번 문제: 모델 구현 확인 (print(model))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(torch.load(config.MODEL_SAVE_PATH, map_location=device))
model.to(device)
model.eval()

print(model)
print(f'\n총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}')

---
## 5번 문제: 모델 훈련 결과 및 텍스트 생성

### 훈련 결과 (실험 #8~#10)

In [ ]:
import json

with open('./models/training_results.json', 'r') as f:
    results = json.load(f)

print('=== 훈련 결과 요약 ===')
print(f'총 에폭: {results["epochs"]}')
print(f'Best Val Loss: {results["best_val_loss"]:.4f} (Epoch {results["best_epoch"]})')
print(f'Final Train Loss: {results["final_train_loss"]:.4f}')
print(f'Final Val Loss: {results["final_val_loss"]:.4f}')

In [ ]:
# 에폭별 손실 출력
print('에폭별 Train Loss / Val Loss')
print('-' * 40)
for i, (tr, vl) in enumerate(zip(results['train_losses'], results['val_losses']), 1):
    print(f'Epoch {i:2d}: Train={tr:.4f}  Val={vl:.4f}')

In [ ]:
# 텍스트 생성 함수
idx2word = tok['idx2word'] if isinstance(tok, dict) else tok.idx2word

def generate(prompt, max_new_tokens=30):
    ids = [word2idx.get(t, 1) for t in prompt.split()]
    x = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            mask = torch.triu(torch.ones(x.size(1), x.size(1)), diagonal=1).bool().to(device)
            logits = model(x, None, mask)
            nt = logits[0, -1, :].argmax().item()
            if idx2word.get(nt) in ['<end>', '<pad>']: break
            x = torch.cat([x, torch.tensor([[nt]]).to(device)], dim=1)
    return ' '.join([idx2word.get(i, '<unk>') for i in x[0].tolist()])

# 텍스트 생성 테스트
test_prompts = ['오늘 기분이', '밥 먹었어', '너무 힘들어', '사랑이 뭔지', '취업이 안 돼']
print('=== GPT 텍스트 생성 결과 ===')
print('-' * 50)
for p in test_prompts:
    print(f'입력: {p}')
    print(f'출력: {generate(p)}')
    print()